# 주문 검사 완료본 (강사용)

두 조건을 모두 채운 예시예요. 처음 실행부터 통과 10건·거부 2건·놓침 0건·합계 112,250원이 나오고, 수량 `0`·`"두"`도 거부돼요. 점검 9가지가 모두 `O`라서 `orders_result.json`에 `"complete": true`가 저장돼요.

학생용 노트북과 알고리즘은 같고, 3단계 두 조건만 채워져 있어요.

순서는 학생용과 같은 6단계예요. 1) 12건 읽기 → 2) `KeyError` 보기 → 3) 조건 읽기 → 4) 분류·비교 → 5) 수량 검사 → 6) 점검·저장.

## 시작하기

1. 위 메뉴에서 **파일 → Drive에 사본 저장**을 눌러요. 내 사본에 답을 남길 수 있어요.
2. 첫 코드 칸 왼쪽의 **▶**를 눌러요. `준비 완료`가 나오면 다음 칸으로 가요.
3. 아래로 순서대로 실행해요. **Shift+Enter**도 같은 칸을 실행하는 방법이에요.

`Cell`은 코드나 설명이 들어 있는 칸이에요. 런타임이 초기화되어 파일이나 변수가 사라졌다면 첫 칸부터 다시 실행해요. 단순히 브라우저를 다시 여는 것과는 달라요.

코드 앞의 `#`는 설명이에요. 실행되지 않아요. `import`는 다른 파일의 이름을 가져오고, 줄 앞의 `!`는 Python 대신 터미널 명령을 실행해요.


## 수정본으로 바꾸려면

GitHub 원본이 바뀌어도 이미 만든 Drive 사본은 자동으로 바뀌지 않아요. 적어 둔 답과 코드를 먼저 저장하세요. [최신 원본 열기](https://colab.research.google.com/github/GoBeromsu/jnu-llmops-precourse-day3/blob/main/solutions/day3_gatekeeper_solution.ipynb)에서 새 Drive 사본을 만든 뒤, 내 답과 수정한 조건만 옮기고 첫 코드 칸부터 실행하세요. 기존 사본을 지우거나 덮어쓰지 않아도 돼요.


In [ ]:
# 필요한 파일을 Colab으로 받아 와요. 실행 상태가 초기화되면 다시 준비해요.
# import는 이름을 가져오고, !는 터미널 명령을 실행하는 표시예요.
import os, sys

if not os.path.isdir("jnu-llmops-precourse-day2"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day2.git
if not os.path.isdir("jnu-llmops-precourse-day3"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day3.git

# 내려받기가 실패했는데 준비됐다고 표시하지 않아요.
for filename in (
    "jnu-llmops-precourse-day2/solution/catalog.py",
    "jnu-llmops-precourse-day2/solution/order.py",
    "jnu-llmops-precourse-day2/solution/pricing.py",
    "jnu-llmops-precourse-day2/solution/receipt.py",
    "jnu-llmops-precourse-day3/data/orders.json",
    "jnu-llmops-precourse-day3/data/expected_day3.json",
    "jnu-llmops-precourse-day3/data/expected_day4.json",
):
    if not os.path.isfile(filename):
        raise FileNotFoundError(f"준비 파일이 없어요: {filename}. 위 다운로드 오류를 강사에게 보여 주세요.")

sys.path.insert(0, "jnu-llmops-precourse-day2/solution")
print("준비 완료 · 왼쪽 파일 탭에 두 폴더가 생겼는지 확인하세요")


## 1단계. 주문 기록 12건을 읽어요

다음 칸을 실행하면 기록 수 `12`와 첫 메뉴 `카페라떼`가 나와요.

`FileNotFoundError`가 나오면 파일을 아직 못 받은 거예요. 첫 준비 칸을 다시 실행하고 이 칸을 이어서 실행해요.

In [ ]:
import json                            # JSON 글자를 파이썬 값으로 바꾸는 도구
from catalog import MENU
from order import Order
from pricing import calculate_bill

# open = 파일을 연다, json.load = 그 글자를 읽어 파이썬 값(목록·딕셔너리)으로 바꾼다
with open("jnu-llmops-precourse-day3/data/orders.json", encoding="utf-8") as f:
    records = json.load(f)

print(len(records))                          # 기록이 몇 건인가
print(records[0]["items"][0]["menu_name"])   # 첫 기록 → items 목록 → 첫 항목 → 이름

## 2단계. 검사 없이 처리하면 어디서 멈출까요?

다음 칸을 실행하면 A01부터 A06까지 금액이 찍히고 마지막에 `KeyError: '녹차라떼'`가 보여요. A07의 `녹차라떼`가 메뉴판에 없기 때문이에요.

이 오류는 이번 실습에서 확인하려고 만든 결과예요. 오류를 화면에만 출력하도록 감싸 두었으니 다음 칸을 이어서 실행하면 돼요.

In [ ]:
import traceback
try:
    for record in records:
        order = Order()
        for item in record["items"]:
            order.add(item["menu_name"], item["quantity"], MENU[item["menu_name"]])
        bill = calculate_bill(order, record["is_student"])
        print(record["order_id"], bill.total)
except Exception:
    traceback.print_exc()

## 3단계. 완성된 두 조건을 읽어요

2번은 메뉴판에 없는 메뉴를, 3번은 정수가 아니거나 1~10 밖인 수량을 거부해요.

다음 두 칸을 실행하면 A07은 `없는 메뉴: 녹차라떼`, A11은 `필수 Key 없음: is_student`로 거부돼요.

In [ ]:
REQUIRED_KEYS = ("order_id", "items", "is_student")

def check_order(record):
    for key in REQUIRED_KEYS:                       # 1. 필수 Key
        if key not in record:
            return False, f"필수 Key 없음: {key}"
    for item in record["items"]:
        if item["menu_name"] not in MENU:           # 2. 허용 메뉴
            return False, f"없는 메뉴: {item['menu_name']}"
        quantity = item["quantity"]
        if type(quantity) is not int or not 1 <= quantity <= 10:   # 3. 수량 범위
            return False, f"수량 범위 밖: {quantity!r}"
    return True, ""


In [ ]:
print(check_order(records[0]))    # 기대 (True, '')
print(check_order(records[6]))    # 기대 (False, '없는 메뉴: 녹차라떼')
print(check_order(records[10]))   # 기대 (False, '필수 Key 없음: is_student')

## 4단계. 12건을 분류하고 기대표와 비교해요

다음 칸은 기록 12건을 통과·거부·놓침으로 나누고, `data/expected_day3.json`의 기대표와 나란히 찍어요.

- 조건을 채우기 전(배포 상태): 통과 10 · 거부 1 · 놓침 1. A07이 문지기를 지나 `KeyError`로 드러나요.
- 두 조건을 채운 뒤: 통과 10 · 거부 2 · 놓침 0.

합계 112,250원은 두 경우 모두 같아요. **합계와 통과 건수만으로는 완성 여부를 알 수 없어요.** 그래서 5단계 수량 검사와 6단계 점검이 필요해요.

`놓침`은 문지기를 통과했지만 계산하다 실패한 주문이에요. 놓침이 남으면 3단계 조건을 고치고 그 칸부터 다시 실행해요.

In [ ]:
accepted, rejected, missed = [], [], []
for record in records:
    ok, reason = check_order(record)
    if not ok:
        rejected.append({"order_id": record["order_id"], "reason": reason})
        continue
    try:
        order = Order()
        for item in record["items"]:
            order.add(item["menu_name"], item["quantity"], MENU[item["menu_name"]])
        bill = calculate_bill(order, record["is_student"])
        accepted.append({"order_id": record["order_id"], "total": bill.total})
    except Exception as e:                     # 문지기가 놓친 것은 여기서 드러납니다
        missed.append({"order_id": record["order_id"], "error": f"{type(e).__name__}: {e}"})

with open("jnu-llmops-precourse-day3/data/expected_day3.json", encoding="utf-8") as f:
    expected = json.load(f)

print("관찰: 통과", len(accepted), "· 거부", len(rejected), "· 놓침", len(missed),
      "· 합계", sum(a["total"] for a in accepted), "원")
print("기대: 통과", expected["accepted"], "· 거부", expected["rejected"], "· 놓침 0",
      "· 합계", expected["accepted_total"], "원")
for r in rejected:
    print("거부", r["order_id"], "-", r["reason"])
for m in missed:
    print("놓침", m["order_id"], "-", m["error"], "← 문지기가 먼저 거부했어야 해요")

## 5단계. 수량의 범위와 타입을 검사해요

기록 12건에는 수량이 잘못된 주문이 없어요. 그래서 수량 조건은 따로 네 건을 만들어 확인해요.

다음 칸을 실행하면 X01(`0`)·X02(`"두"`)·X03(`11`)·X04(`True`)의 검사 결과가 나와요. **네 건 모두 False와 거부 이유가 나와야 해요.** 0과11은 범위 밖이고, 문자열과 참·거짓 값은 수량으로 받지 않아요.

`True`가 나오면 3단계 3번 조건을 고치고, 그 칸부터 여기까지 다시 실행해요.

In [ ]:
QUANTITY_CASES = [
    {"order_id": "X01", "items": [{"menu_name": "카페라떼", "quantity": 0}], "is_student": False},
    {"order_id": "X02", "items": [{"menu_name": "카페라떼", "quantity": "두"}], "is_student": False},
    {"order_id": "X03", "items": [{"menu_name": "카페라떼", "quantity": 11}], "is_student": False},
    {"order_id": "X04", "items": [{"menu_name": "카페라떼", "quantity": True}], "is_student": False},
]

for case in QUANTITY_CASES:
    print(case["order_id"], repr(case["items"][0]["quantity"]), check_order(case))


## 6단계. 점검하고 결과 파일을 저장해요

`save_report`는 지금 정의된 `check_order`로 처음부터 다시 계산해요. 완성본에서는 점검 9가지가 모두 `O`로 찍히고 `orders_result.json`에 `"complete": true`가 저장돼요.

점검에는 수량 `0`·`"두"` 거부까지 들어 있어요. 건수와 합계만 맞아서는 `true`가 되지 않아요.

In [ ]:
def save_report(check, records, path="orders_result.json"):
    """지금 함수로 다시 계산하고, 미완성 기록도 빠짐없이 저장해요."""
    with open("jnu-llmops-precourse-day3/data/expected_day3.json", encoding="utf-8") as f:
        expected = json.load(f)

    accepted, rejected, missed = [], [], []
    for record in records:
        try:
            ok, reason = check(record)
            if not ok:
                rejected.append({"order_id": record["order_id"], "reason": reason})
                continue
            order = Order()
            for item in record["items"]:
                order.add(item["menu_name"], item["quantity"], MENU[item["menu_name"]])
            bill = calculate_bill(order, record["is_student"])
            accepted.append({"order_id": record["order_id"], "total": bill.total})
        except Exception as e:
            missed.append({"order_id": record["order_id"], "error": f"{type(e).__name__}: {e}"})

    quantity_tests = []
    for case in QUANTITY_CASES:
        try:
            ok, reason = check(case)
            test = {"order_id": case["order_id"],
                    "quantity": case["items"][0]["quantity"],
                    "rejected": ok is False and isinstance(reason, str) and bool(reason),
                    "reason": reason}
        except Exception as e:
            # 함수가 오류로 멈춘 것은 올바른 거부 응답이 아니에요.
            test = {"order_id": case["order_id"],
                    "quantity": case["items"][0]["quantity"],
                    "rejected": False, "error": f"{type(e).__name__}: {e}"}
        quantity_tests.append(test)

    accepted_total = sum(a["total"] for a in accepted)
    checks = {
        "입력 12건이 모두 분류됨": len(accepted) + len(rejected) + len(missed) == len(records) == expected["records"],
        "통과 10건": len(accepted) == expected["accepted"],
        "통과 합계 112250원": accepted_total == expected["accepted_total"],
        "거부 2건의 번호와 이유가 기대표와 같음": (
            len(rejected) == expected["rejected"]
            and {r["order_id"]: r["reason"] for r in rejected} == expected["rejected_reasons"]),
        "놓침 0건": len(missed) == 0,
    }
    for test in quantity_tests:
        checks[f"수량 {test['quantity']!r} 거부"] = test["rejected"]
    complete = all(checks.values())

    report = {"input_count": len(records), "accepted": accepted, "rejected": rejected,
              "missed": missed, "accepted_total": accepted_total,
              "quantity_tests": quantity_tests, "checks": checks, "complete": complete}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    for name, ok in checks.items():
        print("O" if ok else "X", name)
    print("complete:", json.dumps(complete))
    print("저장:", path, "·", "완성" if complete else "미완성 — X 항목을 고치고 3단계부터 다시 실행해요")
    return report

report = save_report(check_order, records)


## 확인한 내용을 세 문장으로 남겨요

기대값을 옮겨 적기보다 내 화면에서 본 값을 적어요.

- 입력: 어떤 파일에서 몇 건을 읽었나요?
- 결과: 통과·거부·놓침은 몇 건이고, 수량 검사 네 건은 어떻게 나왔나요?
- 상태: `orders_result.json`의 `complete`는 무엇이고, `X`로 남은 점검이 있나요?